# SeqCredit — Real Data End-to-End

Runs the full pipeline on the Telecel Ghana MoMo transaction table:

1. **Clone** — pull the repo from GitHub  
2. **Install** — pip install the package  
3. **Load** — read raw transactions from Databricks table  
4. **Pipeline** — derive labels + engineer features (Spark), write CSVs  
5. **Benchmark** — 5-fold CV for LR / XGBoost / RF / LightGBM  
6. **Results** — display summary tables  

> **File system note:** outputs are written to `data/` inside the cloned repo.
> If that path is read-only in your workspace, pass an explicit `output_dir`
> to `build_pipeline`, e.g. `output_dir='/dbfs/tmp/seqcredit/'`.

In [ ]:
# ── Cell 1: Clone repo and install ───────────────────────────────────────────
import os

REPO_URL = "https://github.com/attabeezy/seqcredit-model.git"
BRANCH   = "real-data"
REPO_DIR = "/tmp/seqcredit-model"

os.system(f"rm -rf {REPO_DIR}")
os.system(f"git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
%pip install -r requirements.txt
%pip install -e .

In [ ]:
# ── Cell 3: Load raw transactions ────────────────────────────────────────────
df = spark.sql("SELECT * FROM melodatabricks616.default.yara_dump_table")
print(f"Rows: {df.count():,}  |  Columns: {len(df.columns)}")

In [ ]:
# ── Cell 4: Derive labels + engineer features → write user_features.csv / user_labels.csv
from seqcredit_model.real_data_pipeline import build_pipeline

features_df, labels_df = build_pipeline(df)

print("\nLabel distribution:")
print(labels_df["credit_risk_label"].value_counts().sort_index())
print("\nFeature sample (first 3 rows):")
print(features_df.head(3).to_string())

In [ ]:
# ── Cell 5: 5-fold CV benchmark (LR / XGBoost / RF / LightGBM)
# LSTM and HybridLSTM are automatically skipped — no per-user sequence files.
import subprocess, sys

result = subprocess.run(
    [sys.executable, "src/seqcredit_model/run_cv_benchmark.py"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

In [ ]:
# ── Cell 6: Display results
import pandas as pd
from seqcredit_model.config import DATA_DIR

cv_default = pd.read_csv(DATA_DIR / "cv_results_y_default.csv")
cv_bad     = pd.read_csv(DATA_DIR / "cv_results_y_bad.csv")

print("=== y_default (strict default: never repaid) ===")
print(
    cv_default.groupby("model")[["auc_roc", "auc_pr", "f1", "brier", "ece"]]
    .mean().round(4).sort_values("auc_roc", ascending=False).to_string()
)

print("\n=== y_bad (risky or default: penalised + never repaid) ===")
print(
    cv_bad.groupby("model")[["auc_roc", "auc_pr", "f1", "brier", "ece"]]
    .mean().round(4).sort_values("auc_roc", ascending=False).to_string()
)

In [ ]:
# ── Cell 7 (optional): Significance tests
sig = pd.read_csv(DATA_DIR / "significance_tests.csv")
print(sig[["comparison", "target", "metric", "delta_mean", "p_value", "significant"]].to_string())